# Python程序设计（项目实战） · 工单P5《产线数据仓库与ORM》
**第7周 | 2课时(80分钟) | 理实一体**  | 学生实训版

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 复杂查询：多表JOIN+窗口函数（按小时聚合+环比）SQL与ORM双实现；示范讲解 |
| **B 组长** | 标准达成 | 数据字典完整+批量事务 |
| **C 操作员** | 基础完成 | 4表合理+ORM入库+2条查询 |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）


## 一、课前资讯（Predict·预测）
> **预习快问**：一对多关系在表结构上如何表达？

> **课前检查**：模板仓库已克隆更新；导学单与实施计划已提交平台

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 三、任务要求（Make）
**实施步骤**
1. 四表模型+relationship(back_populates)
2. bulk批量事务入库(100/批)
3. 按设备平均vib/告警Top3查询

**验收标准**：及格(D)=`4表合理+ORM入库+2条查询` ；优秀(B+)=`数据字典完整+批量事务`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [2]:
# -*- coding: utf-8 -*-
"""P5 产线数据仓库与ORM —— 框架(SQLAlchemy+SQLite)
数据: 自动模拟(P4 buffer.db存在则可直接导入, 见 TODO-5)
"""
from datetime import datetime

from sqlalchemy import (Boolean, Column, DateTime, Float, ForeignKey,
                        Integer, String, create_engine, func, select)
from sqlalchemy.orm import DeclarativeBase, Session, relationship

DB = "sqlite:///warehouse.db"

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-1: 补充 area(车间) 字段
> - TODO-2: 联合索引 Index('idx_dev_ts',
>   point_id, ts)
> - TODO-3: 种子数据: 3设备×各2测点 + 模拟记录500条(vib随机,
>   5%>4.5触发告警)
> - TODO-4a: 各设备平均vib (JOIN+GROUP BY →
>   func.avg) TODO-4b: 告警Top3设备 TODO-4c:
>   按小时聚合记录数(func.date_trunc不可用→strftime)

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`4表合理+ORM入库+2条查询`，优秀(B+)=`数据字典完整+批量事务`

### Starter 2：`class Base(DeclarativeBase):`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
class Base(DeclarativeBase):
    pass

### Starter 3：`class Device(Base):`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
class Device(Base):
    """设备表"""
    __tablename__ = "devices"
    id = Column(Integer, primary_key=True)
    code = Column(String(16), unique=True, nullable=False)
    name = Column(String(64))
    points = relationship("SensorPoint", back_populates="device")

    # TODO-1: 补充 area(车间) 字段

### Starter 4：`class SensorPoint(Base):`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
class SensorPoint(Base):
    """测点表(设备↔测点 一对多)"""
    __tablename__ = "points"
    id = Column(Integer, primary_key=True)
    device_id = Column(Integer, ForeignKey("devices.id"))
    metric = Column(String(16))          # temp / vib / press
    unit = Column(String(8))
    device = relationship("Device", back_populates="points")

### Starter 5：`class Record(Base):`
（Investigate: 读代码→Modify: 补全TODO）

In [6]:
class Record(Base):
    """采集记录表(高频写入, 索引优化重点)"""
    __tablename__ = "records"
    id = Column(Integer, primary_key=True)
    point_id = Column(Integer, ForeignKey("points.id"))
    ts = Column(DateTime, default=datetime.utcnow)
    value = Column(Float)
    # TODO-2: 联合索引 Index('idx_dev_ts', point_id, ts)

### Starter 6：`class AlarmEvent(Base):`
（Investigate: 读代码→Modify: 补全TODO）

In [7]:
class AlarmEvent(Base):
    """告警事件表"""
    __tablename__ = "alarms"
    id = Column(Integer, primary_key=True)
    device_id = Column(Integer, ForeignKey("devices.id"))
    ts = Column(DateTime, default=datetime.utcnow)
    level = Column(Integer)
    message = Column(String(128))
    resolved = Column(Boolean, default=False)

### Starter 7：`def init_and_seed():`
（Investigate: 读代码→Modify: 补全TODO）

In [8]:
def init_and_seed():
    e = create_engine(DB, echo=False)
    Base.metadata.drop_all(e)
    Base.metadata.create_all(e)
    with Session(e) as s:
        # TODO-3: 种子数据: 3设备×各2测点 + 模拟记录500条(vib随机, 5%>4.5触发告警)
        # 提示: s.add_all([...]); s.commit() 批量用 s.bulk_save_objects
        pass
    return e

### Starter 8：`def queries(e):`
（Investigate: 读代码→Modify: 补全TODO）

In [9]:
def queries(e):
    with Session(e) as s:
        # TODO-4a: 各设备平均vib (JOIN+GROUP BY → func.avg)
        # TODO-4b: 告警Top3设备
        # TODO-4c: 按小时聚合记录数(func.date_trunc不可用→strftime)
        pass

### Starter 9：`def optimize(e):`
（Investigate: 读代码→Modify: 补全TODO）

In [10]:
def optimize(e):
    """TODO-5: 加索引前后查询耗时对比(EXPLAIN QUERY PLAN via raw sql)"""
    import time
    with e.connect() as c:
        t0 = time.time()
        rows = c.execute(select(Record).where(
            Record.point_id == 1).limit(200)).fetchall()
        print("无索引耗时 %.2fms, %d行" % ((time.time() - t0) * 1000, len(rows)))
        # c.execute(text("EXPLAIN QUERY PLAN SELECT ..."))  # 查看是否SCAN
        # c.execute(text("CREATE INDEX idx_pt_ts ON records(point_id, ts)"))
        # 再测一次对比 → 写入工单

### Starter 10：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [11]:
if __name__ == "__main__":
    e = init_and_seed()
    queries(e)
    optimize(e)

无索引耗时 5.03ms, 0行


## 四、提交要求与避坑
- 提交: `学号_姓名_P5.py` 或本notebook(含输出)
- ⚠ DateTime存ISO，别存字符串
- ⚠ bulk_save_objects批量插入比逐条add快10倍+
- ⚠ session用完即with关闭，防连接泄漏